# FACE bipolar cohort: a first tutorial

This notebook shows how to load the FACE BP data with the `facedata` package and how to take a first look at them. It is written for people who have never used this cohort and who do not code much.

You will learn how to:

1. load the data and understand the six tables you get;
2. find a variable and read what it means;
3. look at questionnaires, lab results, neuropsychology, treatments and history at the inclusion visit;
4. follow a few patients over time, and the whole cohort over time;
5. judge how much a value can be trusted;
6. avoid the traps of this cohort;
7. build your own table and save it.

**How to use a notebook.** A notebook is made of cells. Click in a grey cell and press `Shift` + `Enter` to run it: the result appears under it. Run the cells from top to bottom, because each one uses what the cells above it made. You can change a cell and run it again as often as you like; nothing you do here changes the data file.

## Before you start: these are patient data

The results that appear under the cells are patient data, even if the patients have no name.

- **Before you share this notebook or send it to GitHub, clear the results:** menu *Edit*, then *Clear Outputs of All Cells*, then save.
- Never copy patient rows into an e-mail, a chat or a slide. Share counts, means and figures of groups.
- Save your own tables in the folder `data/exports/`. Everything in `data/` stays on this computer.

## 1. Load the cohort

The first cell brings in the tools. The second one reads the extraction: it takes about a minute and a half, so wait for the number in brackets on the left of the cell to appear.

If your extraction file has another name, change it in the first cell.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from facedata import bp

EXTRACTION = "data/bp/BPALL_20261113_2.tsv"        # the file you received
pd.set_option("display.max_columns", 30)

In [ ]:
result = bp.load(EXTRACTION)

`result` holds six tables. You will mostly use the first two.

| Table | One row per | What it holds |
|---|---|---|
| `result.data` | patient visit | everything measured at a visit: scales, lab results, tests, answers |
| `result.patients` | patient | what does not change: sex, diagnosis, family history |
| `result.treatments` | patient and medication | each medication with its start and end dates |
| `result.relatives` | patient and relative | the family history, relative by relative |
| `result.qc` | variable | the catalogue: every variable, its label and how well it is filled |
| `result.flags` | flagged cell | every value that was repaired, doubted or removed, with the reason |

Let us give the first two a short name and count.

In [ ]:
data = result.data
patients = result.patients

print(len(data), "visits of", data["usubjid"].nunique(), "patients")
print(data.shape[1], "columns in data and", patients.shape[1], "in patients")

## 2. What a visit looks like

Every row of `data` starts with the same columns:

- `usubjid`: the patient's number. It links all the tables.
- `visit`: the name of the visit. `screening` comes before inclusion, `V0` is the inclusion visit, `V6_mois` is six months later, `V1_an` one year later, `V2_ans` two years later, and so on.
- `visit_month`: the planned number of months since inclusion (0, 6, 12, 24...). Easier to plot than the names.
- `visit_date`, `siteid` (the centre), `age` (at the visit).

The cell below shows the first rows, with two scores next to these columns.

In [ ]:
data[["usubjid", "visit", "visit_month", "visit_date", "siteid", "age", "madrs_total", "ymrs_total"]].head()

How many visits of each kind are there? `value_counts()` counts the rows for each value, and `.plot.bar()` draws the counts.

In [ ]:
data["visit"].value_counts().plot.bar(figsize=(10, 3), title="Number of visits of each kind")
plt.show()

Most analyses start from the inclusion visit. The next cell keeps only the rows where `visit` is `V0`, and counts the inclusions per year.

**The pattern to remember:** `table[table["column"] == value]` keeps the rows where the column has that value.

In [ ]:
inclusion = data[data["visit"] == "V0"]

inclusion["visit_date"].dt.year.value_counts().sort_index().plot.bar(figsize=(10, 3), title="Inclusion visits per year")
plt.show()

## 3. Who the patients are

Sex, diagnosis and age at inclusion are in `patients`, one row per patient.

In [ ]:
print(patients["sex"].value_counts())
print()
print(patients["diagnosis"].value_counts())

patients["age_at_inclusion"].plot.hist(bins=30, title="Age at inclusion (years)")
plt.show()

To use these columns together with a visit, join the two tables on the patient's number with `merge`. From now on `inclusion` also has the sex and the diagnosis.

In [ ]:
inclusion = inclusion.merge(patients[["usubjid", "sex", "diagnosis"]], on="usubjid")
inclusion.groupby("diagnosis")["age"].agg(["count", "mean"]).round(1)

## 4. Find a variable

There are more than two thousand variables. `result.qc` is their catalogue: the name to type, a label, the instrument it belongs to, the dictionary tab, and `n_valid`, the number of values that can be used.

The cell below searches the labels for a word. Change `"lithium"` to anything: `"sleep"`, `"suicide"`, `"cholesterol"`...

In [ ]:
catalogue = result.qc[["variable", "label", "instrument", "tab", "type", "unit", "level", "n_valid"]]

catalogue[catalogue["label"].str.contains("lithium", case=False)]

The column `level` tells you in which table the variable is: `visit` means `data`, `patient` means `patients`, `list` means `treatments` and `relatives` means `relatives`.

You can also list everything one instrument holds, or count the instruments of each tab.

In [ ]:
print(catalogue.groupby("tab")["instrument"].nunique())

catalogue[catalogue["instrument"] == "fast"]

**Read the notes of an instrument before you use it.** Each instrument has a short text that says what was asked, how the score is computed and what to be careful about. This is where the traps are written.

In [ ]:
from facedata.schema import load_schemas

notes = load_schemas(bp.SCHEMAS)
print(notes["madrs"]["notes"])

## 5. Questionnaires and rating scales

Four scores at the inclusion visit: MADRS (depression, rated by the clinician), YMRS (mania), FAST (functioning, higher is worse) and QIDS (depression, rated by the patient).

`describe()` gives the number of values, the mean, the spread and the quartiles.

In [ ]:
scores = ["madrs_total", "ymrs_total", "fast_total", "qids_total"]

inclusion[scores].describe().round(1)

A histogram shows how the values are spread. These scores are whole numbers, so a few bars look too low or too high only because of where the bars are cut: change  to another number and run the cell again to see.

In [ ]:
inclusion[scores].hist(bins=30, figsize=(10, 6))
plt.show()

Do the clinician and the patient agree on depression? Each dot is one patient at inclusion.

In [ ]:
inclusion.plot.scatter(x="madrs_total", y="qids_total", alpha=0.2, title="Depression: clinician (MADRS) against patient (QIDS)")
plt.show()

inclusion[["madrs_total", "qids_total"]].corr().round(2)

Compare groups with `groupby`: the mean of each score for each diagnosis, then a box plot of the functioning score.

In [ ]:
print(inclusion.groupby("diagnosis")[scores].mean().round(1))

inclusion.boxplot(column="fast_total", by="diagnosis")
plt.show()

## 6. Lab results

Lab results arrive from many laboratories, in several units. The package brings each one to a single unit, shown in the catalogue.

In [ ]:
labs = ["glucose", "crp", "tsh", "creatinine", "lithium"]

catalogue[catalogue["variable"].isin(labs)]

In [ ]:
inclusion[labs].describe().round(2)

**Results below the detection limit.** A laboratory sometimes answers "less than 5" instead of a number. The value then holds the limit (5), and a second column ending in `_cens` says so: `-1` means "the true value is below", `0` means measured, `1` means "above".

In [ ]:
inclusion["crp_cens"].value_counts()

Two pictures: fasting glucose, and the lithium level of the patients who were on lithium on the day of the visit (`on_lithium` is explained in section 8).

In [ ]:
figure, (left, right) = plt.subplots(1, 2, figsize=(11, 3.5))

inclusion["glucose"].plot.hist(bins=40, ax=left, title="Fasting glucose (mmol/L)")
inclusion[inclusion["on_lithium"] == 1]["lithium"].plot.hist(bins=30, ax=right, title="Lithium level of patients on lithium (mmol/L)")
plt.show()

Body mass index, cut into the usual classes with `pd.cut`.

In [ ]:
bmi_class = pd.cut(inclusion["bmi"], [0, 18.5, 25, 30, 100], labels=["under 18.5", "18.5 to 25", "25 to 30", "30 and more"])

bmi_class.value_counts(sort=False).plot.bar(title="Body mass index at inclusion")
plt.show()

## 7. Neuropsychology

The tests are given at inclusion and at some yearly visits, not at every visit.

Many results come as a **z-score**: 0 is the average of people of the same age in the general population, and here higher is always better. A z-score of -1 is one standard deviation below that average.

In [ ]:
print(data[data["tmt_a_time"].notna()]["visit"].value_counts())

tests = ["tmt_a_time_z", "tmt_b_time_z", "wais4_ds_z", "cvlt_total_1_5_z", "stroop_interference_z", "fluency_p_z"]

inclusion[tests].plot.box(figsize=(11, 4), title="Z-scores at inclusion (0 is the population average)")
plt.axhline(0, color="grey")
plt.show()

**Two editions of the same test are never mixed.** The centres moved from WAIS-III to WAIS-IV. The two editions do not count the same way, so each has its own columns (`wais3_...`, `wais4_...`), and `wais_version` says which one was used at a visit. Compare patients inside one edition, or use the z-scores.

In [ ]:
print(data["wais_version"].value_counts())

inclusion.groupby("wais_version")[["wais3_ds_total", "wais4_ds_total"]].agg(["count", "mean"]).round(1)

## 8. Treatments

`result.treatments` has one row per patient and medication, with the name as it was written, the molecule, a group, and the start and end dates.

In [ ]:
treatments = result.treatments
print(len(treatments), "medications of", treatments["usubjid"].nunique(), "patients")

figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
treatments["treatment_group"].value_counts().sort_values().plot.barh(ax=left, title="Medications per group")
treatments["molecule"].value_counts().head(15).sort_values().plot.barh(ax=right, title="The 15 most frequent molecules")
plt.tight_layout()
plt.show()

What you usually want is simpler: was the patient on lithium on the day of the visit? These answers are already in `data`, as columns that start with `on_`: 1 for yes, 0 for no, empty when the patient has no list of medications.

In [ ]:
groups = ["on_lithium", "on_anticonvulsant", "on_antipsychotic_2g", "on_antidepressant", "on_benzodiazepine"]

(inclusion[groups].mean() * 100).round(0).sort_values().plot.barh(title="Patients on each group at inclusion (%)")
plt.show()

## 9. History, comorbidities and family

Yes/no answers are stored as 1 and 0, so the mean of a column is the share of "yes". Always show the number of answers next to it.

These four columns end in `_v0` or `_lifetime`: they are the answers of the inclusion visit, about the whole life (see the traps in section 12).

In [ ]:
answers = ["anxiety_disorder_v0", "substance_disorder_v0", "eating_disorder_v0", "isf05_lifetime"]

table = inclusion[answers].agg(["count", "mean"]).T
table["percent"] = (table["mean"] * 100).round(1)
table[["count", "percent"]]

The family history comes in two forms. The table `relatives` has one row per relative. The columns of `patients` that start with `fdr_` sum it up for the parents, brothers, sisters and children: 1 when one of them has the disorder, 0 when all of them are answered and none has it, and empty when it cannot be told.

In [ ]:
print(result.relatives["kinship"].value_counts())
print()
print(patients["fdr_bipolar"].value_counts(dropna=False))

## 10. Follow patients over time

### A few patients

We look for patients who have a MADRS at six visits or more, and take six of them. `random_state=1` makes it the same six every time; change the number to see others.

The patients are called "Patient 1" to "Patient 6" in the figure, so that no patient number is printed.

In [ ]:
followed = data[data["madrs_total"].notna() & data["visit_month"].notna()]

visits_per_patient = followed.groupby("usubjid").size()
often_seen = visits_per_patient[visits_per_patient >= 6].index.to_series()
chosen = often_seen.sample(6, random_state=1).tolist()

print(len(often_seen), "patients have a MADRS at six visits or more")

In [ ]:
figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for number, patient in enumerate(chosen, start=1):
    visits = followed[followed["usubjid"] == patient].sort_values("visit_month")
    left.plot(visits["visit_month"], visits["madrs_total"], marker="o", label=f"Patient {number}")
    right.plot(visits["visit_month"], visits["ymrs_total"], marker="o")

left.set_title("MADRS (depression)")
right.set_title("YMRS (mania)")
left.set_xlabel("Months since inclusion")
right.set_xlabel("Months since inclusion")
left.legend()
plt.show()

The whole follow-up of one patient as a table: choose the columns you want to read side by side.

In [ ]:
one = followed[followed["usubjid"] == chosen[0]].sort_values("visit_month")

one[["visit", "visit_date", "age", "madrs_total", "ymrs_total", "fast_total", "bmi", "on_lithium"]]

### The whole cohort

The mean score at inclusion and at each yearly visit. Look at `count` first: fewer and fewer patients are seen.

In [ ]:
yearly = data[data["visit"].isin(["V0", "V1_an", "V2_ans", "V3_ans", "V4_ans", "V5_ans"])]

summary = yearly.groupby("visit_month")[["madrs_total", "fast_total"]].agg(["count", "mean"]).round(1)
print(summary)

yearly.groupby("visit_month")[["madrs_total", "fast_total"]].mean().plot(marker="o", title="Mean score at each yearly visit")
plt.xlabel("Months since inclusion")
plt.show()

**Be careful with this figure.** The patients seen at three years are not all the patients included: some left, some were included too recently. If those who stay were less ill from the start, the mean goes down even if nobody improves. The cell below compares, at inclusion, the patients who were seen at three years with the others.

In [ ]:
seen_at_3_years = data[data["visit"] == "V3_ans"]["usubjid"]
inclusion["seen_at_3_years"] = inclusion["usubjid"].isin(seen_at_3_years)

inclusion.groupby("seen_at_3_years")[["age", "madrs_total", "fast_total"]].agg(["count", "mean"]).round(1)

A fairer way to look at change is to compare each patient with themself. `pivot` puts the two visits side by side, one row per patient, and `dropna()` keeps the patients who have both.

In [ ]:
two_visits = data[data["visit"].isin(["V0", "V1_an"])]
side_by_side = two_visits.pivot(index="usubjid", columns="visit", values="fast_total").dropna()

change = side_by_side["V1_an"] - side_by_side["V0"]
print(len(change), "patients have a FAST at inclusion and at one year")
print("Mean change:", round(change.mean(), 1), "(below 0 is an improvement)")

change.plot.hist(bins=40, title="Change of the FAST score between inclusion and one year")
plt.show()

## 11. How much can a value be trusted?

While loading, the package repairs what it can and writes down everything it does in `result.flags`. There are three levels:

- `info`: the value was repaired and can be trusted (a decimal comma, a unit converted, a "less than 5").
- `discrepancy`: the value is kept, but something disagrees with it. Decide for yourself.
- `danger`: the value could not be used. It is empty in `data`, and the text that was typed is kept in the flags.

In [ ]:
flags = result.flags
print(flags["severity"].value_counts())
print()
print(flags[flags["variable"] == "glucose"]["flag"].value_counts())

For glucose, `unit_converted` and `unit_inferred` mean that results written in g/L were brought to mmol/L. Each flag keeps the text as it was typed (`raw`) and the value now delivered (`value`).

The catalogue also gives, for each variable, how many cells carry each kind of flag.

In [ ]:
result.qc[result.qc["variable"].isin(["glucose", "crp", "madrs_total"])][["variable", "n_filled", "n_valid", "n_info", "n_discrepancy", "n_danger", "min", "median", "max"]]

**If you want to be strict**, you can empty the values that carry a `discrepancy` flag before an analysis. The cell below does it for glucose, in a copy of the table.

In [ ]:
doubted = flags[(flags["variable"] == "glucose") & (flags["severity"] == "discrepancy")][["usubjid", "visit"]].drop_duplicates()
doubted["doubted"] = True

strict = data.merge(doubted, on=["usubjid", "visit"], how="left")
strict.loc[strict["doubted"] == True, "glucose"] = float("nan")

print("Glucose values:", data["glucose"].notna().sum(), "before,", strict["glucose"].notna().sum(), "after")

**The quality-control report.** The package can also write a small website with one page per instrument: the notes, the quality figures and four pictures per variable (by year, by centre). It takes about ten minutes. Remove the `#` in the cell below to make it, then open `reports/bp/index.html` in a web browser.

In [ ]:
# bp.report(result, "reports/bp")

## 12. The traps of this cohort

Read this list once; it will save you from the most common mistakes.

1. **`_v0` and `_fu` are two different questions.** For comorbidities, medical history, hospitalisations and suicidal behaviour, the inclusion visit asks about the whole life (`_v0`, `_lifetime`) and the later visits ask about the time since the last visit (`_fu`, `_since_visit`). Never put the two in one column.
2. **Editions and forms have their own columns.** WAIS-III and WAIS-IV, BIS-10 and BIS-12, the long and short ALS, WURS and AIM, DSM-IV and DSM-5. A column such as `wais_version`, `bis_form` or `dsm_version` says which one a visit used.
3. **Empty does not mean "no".** An empty cell means not asked, not answered or not usable. A "no" is written 0.
4. **Sex, diagnosis and family history are in `patients`.** Join them to the visits with `merge`, as in section 3.
5. **Screening visits come before inclusion** and hold almost no measure. Start from `V0`.
6. **Patients leave.** Every figure over time mixes change and departures (section 10).
7. **Centres differ.** They do not include the same patients and do not fill the forms the same way. Look at your variable per centre before you conclude.
8. **The forms changed over the years**, most of all around 2022. Always look at your variable per year. The cell below shows one example.
9. **Extreme values of measures are brought back** to the 1st and 99th percentile (flag `winsorised`). The typed value is kept in the flags.
10. **Everything is a draft until it is reviewed.** The notes of each instrument and the file `docs/bp/PROGRESS.md` list what is known and what is still an open question.

An example of trap 8. The share of patients with a migraine at inclusion seems to double from 2022. It does not: from 2022 the form is mostly filled for the patients who have a neurological disease, so there are far fewer answers (`count`) and almost all the missing ones would be "no".

In [ ]:
by_year = inclusion.groupby(inclusion["visit_date"].dt.year)["migraine_v0"].agg(["count", "mean"]).round(2)
by_year.tail(10)

And trap 7: the same score in each centre. A centre with few patients gives an unstable mean, so look at `count` too.

In [ ]:
inclusion.groupby("siteid")["fast_total"].agg(["count", "mean"]).round(1)

## 13. Build your own table and save it

A typical analysis needs one row per patient with a handful of columns. Choose the visit, choose the columns, join the patient columns, and save.

The file goes to `data/exports/`, which stays on this computer.

In [ ]:
from pathlib import Path

columns = ["usubjid", "visit_date", "siteid", "age", "madrs_total", "ymrs_total", "fast_total", "bmi", "on_lithium", "anxiety_disorder_v0"]

my_table = data[data["visit"] == "V0"][columns]
my_table = my_table.merge(patients[["usubjid", "sex", "diagnosis", "fdr_bipolar"]], on="usubjid")

Path("data/exports").mkdir(exist_ok=True)
my_table.to_csv("data/exports/my_first_table.csv", index=False)

print(len(my_table), "patients and", my_table.shape[1], "columns saved")

A first question with this table: do patients with an anxiety disorder function less well at inclusion?

This is a description, not a proof. For a test or a model that takes age, sex and centre into account, open the saved file in your statistics software (R, SPSS, Jamovi) or ask a statistician.

In [ ]:
print(my_table.groupby("anxiety_disorder_v0")["fast_total"].agg(["count", "mean", "std"]).round(1))

my_table.boxplot(column="fast_total", by="anxiety_disorder_v0")
plt.show()

## 14. Load only what you need

Loading everything takes a minute and a half. When you know which instruments you need, name them: it takes a few seconds. The names are in the column `instrument` of the catalogue.

In [ ]:
small = bp.load(EXTRACTION, instruments=["madrs", "ymrs", "fast"])

print(small.data.shape[1], "columns instead of", data.shape[1])

## Where to go next

- The notes of each instrument (section 4) and the report (section 11).
- `README.md` for the package, and `docs/bp/PROGRESS.md` for every decision taken about the data and every open question.
- The five moves used in this notebook cover most needs: keep rows with `table[table["column"] == value]`, count with `value_counts()`, summarise with `groupby(...).mean()`, join with `merge`, and draw with `.plot`.

**Before you close: clear the results** (menu *Edit*, *Clear Outputs of All Cells*) if this notebook will be shared.